# Module 08 — AI Workbook Companion (N-body optimization)

A lightweight companion to the [GPU challenge notebook](../GPU%20challenge.ipynb)
and [01-nbody.cu](../01-nbody.cu). It does **not** replace the challenge — it adds
the supervision layer from [Module 11](../../11/README.md) to Module 08's topic:
**optimizing an all-pairs N-body simulation without breaking its physics.**

New here? Read [Module 11's README](../../11/README.md) first — it explains the
5-phase loop, what a "CPU reference" is, and the known ways AI gets CUDA wrong.

## The loop, applied to N-body

Your task: accelerate the all-pairs gravitational force computation and the
position integration on the GPU. You may have an AI write the kernels. Your job
is everything around it.

1. **SPECIFY** — Contract: body layout, launch configuration, the two distinct
   phases (compute forces → *then* integrate positions), which arrays are
   read vs written in each phase, and the metric (billions of interactions/sec).
2. **GENERATE** — Ask the AI for the kernels. Keep them in their own file.
3. **PREDICT** — *Before running:* this is compute-bound (O(N²) work over N
   bodies) — good for the GPU. The correctness risk is the **phase separation**:
   every force must be computed from the *old* positions before *any* position is
   updated. Does the generated code respect that?
4. **VERIFY** — Use [verify_nbody.cu](./verify_nbody.cu): CPU reference plus a
   harness *stub*. You launch the kernels and compare final positions/velocities
   against the reference.
5. **DIAGNOSE** — Profile. Is it compute-bound as predicted? Would shared-memory
   tiling of the body positions help?

## The adversarial exercise

[adversarial_nbody_buggy.cu](./adversarial_nbody_buggy.cu) is a kernel "an AI
wrote for you." It fuses force computation and position integration into a
**single** kernel, so a thread can update its own position while other threads
are still reading it to compute their forces. That is a race across the whole
grid — the physics is silently wrong, and "interactions/sec" can even look
*better* because the kernel does less synchronized work.

Your job:

1. Read it and predict why fusing the phases is unsafe *before* running.
2. Build and run it against the CPU reference. Note the mismatch.
3. State the exact bug (which write races which read) and the fix (separate the
   phases into two kernels / two launches).
4. Prove the fix: final state matches the CPU reference within tolerance.

Could an AI catch this? Sometimes — but a whole-grid race produces no compiler
error, sometimes a plausible-looking trajectory, and a *faster* headline number,
so it slips past a "it got faster and looks fine" review. A comparison against a
CPU reference catches it. That is your job.

A worked solution and explanation are in [solution.ipynb](./solution.ipynb). Try it
yourself first.

## Reflection prompt (write this down)

- Why must *all* forces be computed before *any* position is integrated? What
  exactly races if you fuse them?
- The buggy version can report higher "interactions/sec." Why is a performance
  number meaningless without the correctness gate?
- N-body is O(N²) compute over O(N) data — compute-bound. What optimization does
  that point to (shared-memory tiling), and how much would you expect it to buy?


In [ ]:
# Prerequisites: verify the GPU toolchain before running this workbook.
import pathlib, runpy
_here = pathlib.Path.cwd()
for _d in (_here, *_here.parents):
    if (_d / "check_gpu.py").exists():
        runpy.run_path(str(_d / "check_gpu.py"), run_name="__main__")
        break
else:
    print("check_gpu.py not found - run `python check_gpu.py` from the repo root.")

## Step 1 - Reproduce the bug

Build and run the problem program [adversarial_nbody_buggy.cu](./adversarial_nbody_buggy.cu). Read it first and predict what will happen before you run this cell.

In [ ]:
!nvcc -arch=native adversarial_nbody_buggy.cu -o buggy && ./buggy

## Step 2 - Run your verification harness

[verify_nbody.cu](./verify_nbody.cu) ships a CPU reference and a PASS/FAIL gate, but it **defaults to FAIL** until you complete the `TODO` sections in it (launch, error check, synchronization). Edit the file, then re-run this cell until it prints `PASS`.

In [ ]:
!nvcc -arch=native verify_nbody.cu -o verify && ./verify

## Next

Once your harness prints `PASS`, open **[solution.ipynb](./solution.ipynb)** to compare your fix with the worked answer and explanation. See [Module 11](../../11/README.md) for the method behind this workbook.